<a href="https://colab.research.google.com/github/smerashanmugan/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_tarts = cat[cat['product_name'].str.contains('pop', case=False, na=False)]
print(len(pop_tarts))
print(pop_tarts)

3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


I filtered the product names to find anything with "pop" in it, ignoring capitalization. I did this to see how many Pop-Tart products showed up and if they were listed under different names.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = cat[
    cat['product_name'].str.contains('strawberry', case=False, na=False)
    & cat['product_name'].str.contains('pop', case=False, na=False)
]

print(strawberry[['sku', 'product_name']])

not_strawberry = pop_tarts[
    ~pop_tarts['product_name'].str.contains('strawberry', case=False, na=False)
]

print(not_strawberry[['sku', 'product_name']])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** None in the sample data. All three Pop-Tart rows are strawberry, so there are no other flavors that need to stay separate.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
sku_mapping = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

print(cat[['sku', 'product_name', 'canonical_sku']])

             sku          product_name canonical_sku
0          PT-12  pop-tarts strawberry         PT-12
1     SKU#459812   Strawberry Pop-Tart         PT-12
2  POPTART-STRAW  Pop-Tarts Strawberry         PT-12


I created a dictionary called sku_mapping to group the different strawberry Pop-Tart SKUs under one SKU, PT-12. Then I used .map() to apply the mapping to the original SKUs. I added .fillna(cat['sku']) so that any SKUs not in the dictionary would keep their original values instead of becoming NaN. Finally, I printed the columns to check that the mapping worked.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print(cat['category'].value_counts())

cat['category_clean'] = cat['category'].str.strip().str.title()

print(cat['category_clean'].value_counts())

category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
category_clean
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


I used .value_counts() to see the different categories and how many times each one appeared before cleaning. Then I used .str.strip() to remove extra spaces and .str.title() to make the capitalization consistent. Finally, I used .value_counts() again to see the categories after cleaning. I noticed that Breakfast and Breakfast Foods were still separate even though they mean the same thing, so those would need to be grouped together.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def normalize_name(name):
    name = str(name).lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

groups = cat.groupby('normalized_name').filter(
    lambda x: x['sku'].nunique() > 1
)

print(groups[['sku', 'product_name', 'normalized_name']])

             sku          product_name      normalized_name
0          PT-12  pop-tarts strawberry  poptarts strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry  poptarts strawberry


I imported re to help remove punctuation from the product names. Then I created a function called normalize_name that makes the names lowercase, removes punctuation, and sorts the words so different word orders can still match. I used .apply() to run the function on each product name. After that, I used .groupby() to group matching names and .filter() to find groups with more than one SKU. Finally, I printed the results to see which products might be duplicates.


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all()
assert len(cat) == len(cat['canonical_sku'])

print("Mapping is safe!")

Mapping is safe!


I used assert with .notna().all() to check that there were no missing values in the canonical_sku column. Then I used len() to check that the number of rows stayed the same after mapping. I did this to make sure the mapping didn't lose any SKUs or create extra rows.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) I only had Pop-Tarts in my sample data but if I had the full catalog, I would look for products with similar names but different SKUs and group them together if they were the same product.
b) For example, I wouldn't be sure if different Pop-Tart flavors should be grouped together, so I would ask the product or inventory manager first. I think that's where it becomes more of a business decision because even though the products are similar, they aren't necessarily the same.